In [1]:
!git clone https://github.com/IzherBajeed/internship-recommendation-project.git

Cloning into 'internship-recommendation-project'...


In [2]:
%cd internship-recommendation-project

/content/internship-recommendation-project


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import re


In [ ]:
applicant = pd.read_csv('applicant_dataset.csv')
internship = pd.read_csv('internship.csv')

In [ ]:
internship.head()

,internship_title,company_name,location,start_date,duration,stipend
0,Java Development,SunbaseData,Work From Home,Immediately,6 Months,"₹ 30,000 /month"
1,Accounting and Finance,DAKSM & Co. LLP,Noida,Immediately,6 Months,"₹ 5,000-10,000 /month"
2,Sales & Digital Marketing,Bharat Natural Elements Private Limited,Bangalore,Immediately,6 Months,"₹ 5,000 /month"
3,Social Entrepreneurship,Hamari Pahchan NGO,Work From Home,Immediately,6 Months,Unpaid
4,Videography & Photography,Esquare Lifestyle,Bangalore,Immediately,6 Months,"₹ 12,000 /month"


In [ ]:
internship.isnull().sum()

,0
internship_title,0
company_name,0
location,0
start_date,0
duration,0
stipend,0


In [ ]:
internship.duplicated().sum()

np.int64(36)

In [ ]:
internship.drop_duplicates().sum()

,0
internship_title,Java DevelopmentAccounting and FinanceSales & ...
company_name,SunbaseDataDAKSM & Co. LLPBharat Natural Eleme...
location,Work From HomeNoidaBangaloreWork From HomeBang...
start_date,ImmediatelyImmediatelyImmediatelyImmediatelyIm...
duration,6 Months6 Months6 Months6 Months6 Months6 Mont...
stipend,"₹ 30,000 /month₹ 5,000-10,000 /month₹ 5,000 /m..."


In [ ]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9, ]', '', text)
    return text

applicant['Skills'] = applicant['Skills'].apply(clean_text)
internship['internship_title'] = internship['internship_title'].apply(clean_text)

In [ ]:

internship.head()

,internship_title,company_name,location,start_date,duration,stipend
0,java development,SunbaseData,Work From Home,Immediately,6 Months,"₹ 30,000 /month"
1,accounting and finance,DAKSM & Co. LLP,Noida,Immediately,6 Months,"₹ 5,000-10,000 /month"
2,sales digital marketing,Bharat Natural Elements Private Limited,Bangalore,Immediately,6 Months,"₹ 5,000 /month"
3,social entrepreneurship,Hamari Pahchan NGO,Work From Home,Immediately,6 Months,Unpaid
4,videography photography,Esquare Lifestyle,Bangalore,Immediately,6 Months,"₹ 12,000 /month"


In [ ]:
applicant.head()

,ApplicantID,Name,Education,Skills,PreferredDomain,PreferredLocation
0,A001,Rahul Sharma,B.Tech CSE,"python, machine learning, sql",Data Science,Bangalore
1,A002,Priya Verma,MBA Marketing,"communication, seo, analytics",Marketing,Delhi
2,A003,Arjun Patel,B.Com,"accounting, excel, tally",Finance,Mumbai
3,A004,Neha Singh,B.Tech IT,"java, web development, react",Software Development,Hyderabad
4,A005,Sameer Khan,B.Sc Statistics,"r, python, data analysis",Data Analytics,Pune


In [ ]:
!pip install sentence-transformers



In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd


In [ ]:
model = SentenceTransformer('all-minilM-l6-V2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Build profile text for embeddings
applicant['ProfileText'] = (
    applicant['Skills'] + " | Location: " + applicant['PreferredLocation']
)

internship['ProfileText'] = (
    internship['internship_title'] + " | Location: " + internship['location']
)

# Encode
applicant_embeddings = model.encode(applicant['ProfileText'].tolist(), convert_to_tensor=False)
internship_embeddings = model.encode(internship['ProfileText'].tolist(), convert_to_tensor=False)

# Compute cosine similarity
similarity_matrix = cosine_similarity(applicant_embeddings, internship_embeddings)


In [ ]:
 def recommend_embeddings(applicant_id, top_n=5):
    # Find applicant index
    idx = applicant[applicant['ApplicantID'] == applicant_id].index[0]

    # Get similarity scores for this applicant
    sim_scores = list(enumerate(similarity_matrix[idx]))

    # Sort by similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[:top_n]

    # Pick internships
    recommended = internship.iloc[[i[0] for i in sim_scores]].copy()
    recommended["Score"] = [i[1] for i in sim_scores]

    return recommended[['internship_title', 'location', 'Score']]


In [ ]:
recommend_embeddings("A004",top_n=4)

,internship_title,location,Score
5609,react native development,Hyderabad,0.807197
2436,web development,Hyderabad,0.773360
2554,javascript development,Hyderabad,0.764852
4622,web engineering,Hyderabad,0.759809


In [ ]:
recommend_embeddings("A003",top_n=8)

,internship_title,location,Score
5421,accounting tally,Mumbai,0.900631
4712,accounting,Mumbai,0.853422
5882,accounting,Mumbai,0.853422
1962,finance accounting,Mumbai,0.793554
1456,accounting bookkeeping,Mumbai,0.755962
3957,accounting bookkeeping,Mumbai,0.755962
1522,accounting tally,Pune,0.743544
3236,business analytics,Mumbai,0.723403


In [ ]:
import pickle

# suppose your trained model variable is called "model"
with open("recommend_embeddings", "wb") as f:
    pickle.dump(model, f)


In [ ]:
# backend/app.py
from flask import Flask, request, jsonify
import pandas as pd
import pickle

# Initialize Flask app
app = Flask(__name__)

# Load your trained model
model = pickle.load(open("recommend_embeddings", "rb"))

# Load internships dataset (used for matching)
internship = pd.read_csv("internship.csv")

# Example function to generate recommendations
def recommend_with_learning(applicant_data, top_n=5):
    sim_scores = []  # here you will recompute embedding similarity (step from earlier)

    # Example: fake loop (replace with your similarity calculation)
    for i, row in internship.iterrows():
        domain_match = 1 if applicant_data['PreferredDomain'].lower() in str(row['domain']).lower() else 0
        location_match = 1 if applicant_data['PreferredLocation'].lower() in str(row['location']).lower() else 0
        sim = 0.75  # placeholder similarity (replace with embeddings)

        # Predict probability using model
        features = [[sim, domain_match, location_match]]
        prob = model.predict_proba(features)[0][1]

        sim_scores.append((i, prob))

    # Sort and select top_n
    results = sorted(sim_scores, key=lambda x: x[1], reverse=True)[:top_n]
    recommended = internship.iloc[[i[0] for i in results]].copy()
    recommended["Score"] = [i[1] for i in results]

    return recommended[['internship_title', 'domain', 'location', 'Score']].to_dict(orient="records")

# API endpoint
@app.route('/recommend', methods=['POST'])
def recommend():
    data = request.json  # Applicant input from frontend
    recommendations = recommend_with_learning(data, top_n=5)
    return jsonify(recommendations)

if __name__ == '__main__':
    app.run(debug=True)


 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)


In [ ]:
internship.tail()
